In [48]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder , OneHotEncoder , StandardScaler
from sklearn.model_selection import train_test_split

import pickle

import tensorflow as tf
from tensorflow.keras.layers import Dense ,Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

In [17]:
df = pd.read_csv('Churn_Modelling.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [18]:
df.drop(columns=['RowNumber','CustomerId','Surname'],inplace = True)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
label_for_gender = LabelEncoder()
df['Gender'] = label_for_gender.fit_transform(df['Gender'])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
one_hot_for_geography = OneHotEncoder()
geo_encoder = one_hot_for_geography.fit_transform(df[['Geography']]).toarray()
geo_encoder = pd.DataFrame(geo_encoder,columns = one_hot_for_geography.get_feature_names_out() )
geo_encoder

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [33]:
data = pd.concat([df.drop(columns=['Geography']),geo_encoder],axis=1)

In [34]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [37]:
X = data.drop(columns=['Exited'])
y = data['Exited']

X_train , X_test ,y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [38]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [62]:
#dump all file 
with open('label_for_gender.pkl','wb') as f:
    pickle.dump(label_for_gender,f)


with open('one_hot_for_geography.pkl','wb') as f:
    pickle.dump(one_hot_for_geography,f)


with open('scaler.pkl','wb') as f:
    pickle.dump(scaler,f)

In [43]:
X.shape[1]

12

In [50]:
model = Sequential([
    Input(shape=(X.shape[1],)),
    Dense(64,activation='relu'),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [54]:
model.compile(
    loss = 'binary_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)

In [55]:
log_dir="logs/fit/"
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [56]:
history=model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6664 - loss: 621.9169 - val_accuracy: 0.7025 - val_loss: 110.0288
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6752 - loss: 95.0032 - val_accuracy: 0.7100 - val_loss: 74.0016
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6744 - loss: 85.8227 - val_accuracy: 0.7220 - val_loss: 32.8825
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.6814 - loss: 97.7463 - val_accuracy: 0.8035 - val_loss: 228.2160
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6759 - loss: 97.3526 - val_accuracy: 0.8055 - val_loss: 34.0138
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6781 - loss: 71.7946 - val_accuracy: 0.5575 - val_loss: 106.6971
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6780 - loss: 69.4906 - val_accuracy: 0.8035 - val_loss: 164.3066
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6839 - loss

In [61]:
model.save('model.h5')

In [59]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [60]:
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 1696), started 0:01:16 ago. (Use '!kill 1696' to kill it.)

In [63]:
geography='France'
geo_encoder = one_hot_for_geography.transform([[geography]]).toarray()
geo_encoder = pd.DataFrame(geo_encoder,columns = one_hot_for_geography.get_feature_names_out() )

c:\Users\ASHISH\Desktop\Langchain\myenv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [64]:
geo_encoder

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [74]:
data = {
    'CreditScore':745,
    'Gender':label_for_gender.transform(['Male'])[0],
    'Age':43,
    'Tenure':2,
    'Balance':7652,
    'NumOfProducts':1,
    'HasCrCard':1,
    'IsActiveMember': 1,
    'EstimatedSalary': 765

}

data = pd.DataFrame([data])


geo_encoder = one_hot_for_geography.transform([[geography]]).toarray()
geo_encoder = pd.DataFrame(geo_encoder,columns = one_hot_for_geography.get_feature_names_out() )

data = pd.concat([data,geo_encoder],axis=1)



c:\Users\ASHISH\Desktop\Langchain\myenv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [75]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,745,1,43,2,7652,1,1,1,765,1.0,0.0,0.0
